# 📘 W3 Python Lab — 뉴스 센티먼트 + Claude 첫 호출

**n8n W3 강의의 Python 버전.** Alpha Vantage 뉴스 + Anthropic SDK 직접 사용.

## 학습 목표
- Alpha Vantage NEWS_SENTIMENT API 호출
- Anthropic Python SDK로 Claude Haiku 호출
- 뉴스 데이터 + Claude 분석 결합
- 구조화된 JSON 출력 강제

## 🛠 사전 준비

```bash
pip install anthropic requests python-dotenv
```

`.env`:
```
ALPHA_VANTAGE_KEY=your_key
ANTHROPIC_API_KEY=sk-ant-...
```

**API 발급:**
- Alpha Vantage 무료: https://www.alphavantage.co/support/#api-key (25/day)
- Anthropic: https://console.anthropic.com/ ($5 크레딧으로 수업 충분)

> 💡 **n8n과 비교:** n8n의 AI Agent 노드가 내부적으로 하는 일을 Python에서 직접 제어. 프롬프트 엔지니어링과 토큰 사용을 세밀하게 볼 수 있습니다.

## 1. Alpha Vantage 뉴스 수집

In [ ]:
import os
import requests
import json
from dotenv import load_dotenv
from datetime import datetime, timedelta

load_dotenv()
AV_KEY = os.getenv('ALPHA_VANTAGE_KEY')

def fetch_news(ticker: str, hours_back: int = 24, limit: int = 10) -> list:
    """Alpha Vantage에서 종목 관련 뉴스 + 센티먼트 스코어 조회.
    
    sentiment_score: -0.35 ~ 0.35 (음수 부정, 양수 긍정)
    relevance_score: 0 ~ 1 (기사가 해당 ticker에 얼마나 관련)
    """
    time_from = (datetime.now() - timedelta(hours=hours_back)).strftime('%Y%m%dT%H%M')
    
    url = 'https://www.alphavantage.co/query'
    params = {
        'function': 'NEWS_SENTIMENT',
        'tickers': ticker,
        'time_from': time_from,
        'limit': limit,
        'sort': 'LATEST',
        'apikey': AV_KEY
    }
    
    resp = requests.get(url, params=params, timeout=15)
    resp.raise_for_status()
    data = resp.json()
    
    # 해당 ticker 관련 점수만 추출
    articles = []
    for item in data.get('feed', []):
        ticker_scores = {t['ticker']: t for t in item.get('ticker_sentiment', [])}
        if ticker in ticker_scores:
            articles.append({
                'title': item['title'],
                'url': item['url'],
                'source': item['source'],
                'published': item['time_published'],
                'overall_sentiment': float(item['overall_sentiment_score']),
                'ticker_sentiment': float(ticker_scores[ticker]['ticker_sentiment_score']),
                'relevance': float(ticker_scores[ticker]['relevance_score']),
                'summary': item.get('summary', '')[:300]
            })
    
    # 관련도 0.3 이상만
    return [a for a in articles if a['relevance'] >= 0.3]

# 테스트
news = fetch_news('AAPL', hours_back=48)
print(f'관련 기사 {len(news)}건 수집')
for a in news[:3]:
    print(f"\n• [{a['source']}] {a['title'][:70]}...")
    print(f"  sentiment={a['ticker_sentiment']:+.2f}, relevance={a['relevance']:.2f}")

## 2. 센티먼트 스코어 집계

In [ ]:
def aggregate_sentiment(articles: list) -> dict:
    """뉴스 리스트에서 평균 센티먼트 + 분포 계산."""
    if not articles:
        return {'count': 0, 'avg': 0, 'label': 'NO_DATA'}
    
    scores = [a['ticker_sentiment'] for a in articles]
    avg = sum(scores) / len(scores)
    
    # 라벨링 (Alpha Vantage 기준)
    if avg > 0.15:
        label = 'BULLISH'
    elif avg > 0.05:
        label = 'SOMEWHAT_BULLISH'
    elif avg > -0.05:
        label = 'NEUTRAL'
    elif avg > -0.15:
        label = 'SOMEWHAT_BEARISH'
    else:
        label = 'BEARISH'
    
    # 분포
    positive = sum(1 for s in scores if s > 0.05)
    negative = sum(1 for s in scores if s < -0.05)
    neutral = len(scores) - positive - negative
    
    return {
        'count': len(articles),
        'avg': round(avg, 3),
        'label': label,
        'positive': positive,
        'negative': negative,
        'neutral': neutral
    }

agg = aggregate_sentiment(news)
print(json.dumps(agg, indent=2))

## 3. Claude Haiku 첫 호출

Anthropic SDK 기본 사용법. 뉴스 여러 개를 Claude가 종합 요약 + 리스크 플래그.

In [ ]:
import anthropic

client = anthropic.Anthropic()  # ANTHROPIC_API_KEY 환경변수 자동 인식

def claude_summarize_news(ticker: str, articles: list) -> dict:
    """Claude Haiku로 뉴스 종합 요약 + JSON 구조화 출력."""
    
    # 뉴스 텍스트 준비 (상위 5개만)
    news_text = '\n\n'.join([
        f"[{a['source']}] {a['title']}\n요약: {a['summary']}\n센티먼트: {a['ticker_sentiment']:+.2f}"
        for a in articles[:5]
    ])
    
    system_msg = (
        '당신은 금융 뉴스 분석 어시스턴트입니다. '
        '오직 JSON 형식으로만 응답하세요. 설명이나 마크다운 사용 금지.'
    )
    
    user_msg = f"""다음은 {ticker} 관련 최근 뉴스입니다:

{news_text}

다음 JSON 스키마로 종합 분석을 출력하세요:
{{
  "overall_tone": "POSITIVE" | "NEGATIVE" | "NEUTRAL" | "MIXED",
  "key_themes": ["주요 테마 1-2문장씩", ...],
  "risk_flags": ["발견된 리스크 1-2문장씩", ...],
  "catalysts": ["긍정적 촉매 요인 1-2문장씩", ...],
  "one_line_summary": "한 문장 종합"
}}"""
    
    response = client.messages.create(
        model='claude-haiku-4-5-20251001',
        max_tokens=800,
        system=system_msg,
        messages=[{'role': 'user', 'content': user_msg}]
    )
    
    # JSON 파싱
    text = response.content[0].text.strip()
    # ```json``` 제거 (혹시 섞여 있으면)
    text = text.replace('```json', '').replace('```', '').strip()
    
    result = json.loads(text)
    
    # 메타데이터 추가
    result['_meta'] = {
        'model': response.model,
        'input_tokens': response.usage.input_tokens,
        'output_tokens': response.usage.output_tokens,
        'cost_usd': round(
            response.usage.input_tokens * 1e-6 + 
            response.usage.output_tokens * 5e-6, 4
        )
    }
    
    return result

analysis = claude_summarize_news('AAPL', news)
print(json.dumps(analysis, indent=2, ensure_ascii=False))

## 4. W2 기술적 지표와 결합 — 2차원 판단

W2 RSI/MA/BB 점수 + W3 뉴스 센티먼트를 Claude가 종합.

In [ ]:
import yfinance as yf
import pandas as pd

# W2 함수 재사용 (간략 버전)
def calc_rsi(close: pd.Series, period: int = 14) -> pd.Series:
    delta = close.diff()
    gain = delta.where(delta > 0, 0)
    loss = -delta.where(delta < 0, 0)
    avg_gain = gain.ewm(alpha=1/period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1/period, adjust=False).mean()
    return 100 - (100 / (1 + avg_gain / avg_loss))

def claude_combined_verdict(ticker: str) -> dict:
    """뉴스 + 기술적 지표를 Claude가 종합해 최종 verdict."""
    
    # 1) 기술적 지표
    df = yf.Ticker(ticker).history(period='3mo')
    df['RSI'] = calc_rsi(df['Close'])
    df['MA20'] = df['Close'].rolling(20).mean()
    df['MA60'] = df['Close'].rolling(60).mean()
    
    latest = df.iloc[-1]
    technicals = {
        'price': round(latest['Close'], 2),
        'rsi14': round(latest['RSI'], 1),
        'above_ma20': latest['Close'] > latest['MA20'],
        'above_ma60': latest['Close'] > latest['MA60'],
        'week_change': round((latest['Close'] / df.iloc[-5]['Close'] - 1) * 100, 2)
    }
    
    # 2) 뉴스
    articles = fetch_news(ticker, hours_back=48)
    news_agg = aggregate_sentiment(articles)
    
    # 3) Claude가 종합
    system_msg = (
        '당신은 투자 리서치 어시스턴트입니다. '
        '기술적 지표와 뉴스 센티먼트를 모두 고려해 균형 잡힌 판단을 내립니다. '
        '확신이 없으면 WATCH나 HOLD를 택합니다. JSON만 출력하세요.'
    )
    
    user_msg = f"""티커: {ticker}

📊 기술적 지표:
{json.dumps(technicals, indent=2)}

📰 뉴스 센티먼트 (최근 48시간):
{json.dumps(news_agg, indent=2)}

주요 뉴스 제목 (상위 3):
{json.dumps([a['title'] for a in articles[:3]], indent=2, ensure_ascii=False)}

두 가지를 모두 고려해 다음 JSON으로 답변:
{{
  "verdict": "BUY" | "WATCH" | "HOLD" | "AVOID",
  "confidence": 1-5,
  "technical_view": "기술적 지표 한 줄 해석",
  "news_view": "뉴스 한 줄 해석",
  "alignment": "ALIGNED" | "DIVERGENT",
  "reasoning": "2-3문장 최종 근거"
}}"""
    
    resp = client.messages.create(
        model='claude-haiku-4-5-20251001',
        max_tokens=500,
        system=system_msg,
        messages=[{'role': 'user', 'content': user_msg}]
    )
    
    text = resp.content[0].text.strip().replace('```json', '').replace('```', '').strip()
    result = json.loads(text)
    result['ticker'] = ticker
    result['_technicals'] = technicals
    result['_news'] = news_agg
    return result

verdict = claude_combined_verdict('AAPL')
print(json.dumps(verdict, indent=2, ensure_ascii=False))

## 5. 프롬프트 비용 관리 — 토큰 로깅

Claude API는 토큰당 과금. 어느 프롬프트에 얼마 썼는지 CSV에 누적해야 이상 지출을 잡을 수 있습니다.

In [ ]:
from pathlib import Path

def log_claude_usage(task: str, response, filepath: str = 'claude_usage.csv'):
    """모든 Claude 호출을 CSV에 기록."""
    # Haiku 4.5 단가 (참고: 실제 단가는 Anthropic 공식 문서 확인)
    INPUT_COST_PER_TOKEN = 1e-6   # $1 / 1M tokens
    OUTPUT_COST_PER_TOKEN = 5e-6  # $5 / 1M tokens
    
    cost = (response.usage.input_tokens * INPUT_COST_PER_TOKEN + 
            response.usage.output_tokens * OUTPUT_COST_PER_TOKEN)
    
    row = {
        'timestamp': datetime.now().isoformat(),
        'task': task,
        'model': response.model,
        'input_tokens': response.usage.input_tokens,
        'output_tokens': response.usage.output_tokens,
        'cost_usd': round(cost, 6)
    }
    
    df = pd.DataFrame([row])
    path = Path(filepath)
    if path.exists():
        df.to_csv(path, mode='a', header=False, index=False)
    else:
        df.to_csv(path, index=False)
    
    return cost

# 사용량 통계
def usage_summary(filepath: str = 'claude_usage.csv'):
    """누적 Claude 사용량 요약."""
    if not Path(filepath).exists():
        return 'No usage yet'
    
    df = pd.read_csv(filepath)
    print(f'Total calls: {len(df)}')
    print(f'Total cost: ${df["cost_usd"].sum():.4f}')
    print(f'Avg cost per call: ${df["cost_usd"].mean():.4f}')
    print(f'\nBy task:')
    print(df.groupby('task')['cost_usd'].agg(['count', 'sum', 'mean']))

# 사용 예시는 실제 호출 후
print('💡 모든 claude.messages.create() 호출 직후 log_claude_usage() 호출해 누적 비용 관리')

## 🎯 과제

1. **종목별 비교 리포트** — AAPL, MSFT, NVDA 3개에 대해 `claude_combined_verdict` 실행 → 표로 비교
2. **프롬프트 A/B 테스트** — 동일 뉴스를 2가지 다른 프롬프트로 Claude에 입력 → 결과 품질 비교
3. **한국 종목 확장** — Alpha Vantage는 한국 종목 커버리지 제한적 → 대안으로 네이버 금융 크롤링 (BeautifulSoup) 시도
4. **배치 처리** — 포트폴리오 10종목을 순차 실행 + 전체 비용 합산 로깅

## 🔄 n8n vs Python 비교

| 작업 | n8n | Python |
|---|---|---|
| Claude 호출 | AI Agent 노드 | `client.messages.create()` |
| System Message | 노드 설정 필드 | `system=` 파라미터 |
| JSON 출력 강제 | Structured Output Parser | 프롬프트로 명시 + `json.loads()` |
| 토큰 추적 | 수동 (Console) | `response.usage` 속성 자동 |
| 에러 복구 | Error Trigger | `try/except + retry` |

**Python의 장점:** 토큰 사용량, 비용, 응답 구조를 **프로그래밍으로 완벽 제어**. 실험과 최적화에 유리.

**n8n의 장점:** 비개발자도 즉시 사용 가능. 스케줄·알림·DB 연동이 시각적.